# Portuguese Pre-processing

In [1]:
import glob
import random
import tqdm
import nltk
import unicodedata
import re
import contractions
import nltk
import spacy
import glob

# Ensure necessary NLTK data is downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Load SpaCy English model
# nlp = spacy.load('en_core_web_lg')
nlp = spacy.load("pt_core_news_lg")

In [2]:
preso_stf = glob.glob("../data/datasets/STF_HC/train/raw/Preso/*.txt")
solto_stf = glob.glob("../data/datasets/STF_HC/train/raw/Solto/*.txt")

stf_train = preso_stf + solto_stf

random.shuffle(stf_train)

In [3]:
solto_raw_dataset = [open(f).read() for f in solto_stf]
preso_raw_dataset = [open(f).read() for f in preso_stf]


In [4]:
# Import necessary libraries
import re
import unicodedata
import spacy
from nltk.tokenize import sent_tokenize
import nltk

# --- Setup: Download necessary NLP models ---
# The following lines ensure that the required NLTK and spaCy models are available.
# It's good practice to handle this programmatically.
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    print("Downloading 'punkt' from NLTK...")
    nltk.download('punkt')

try:
    # Load the Portuguese spaCy model. We use the large model for better accuracy
    # on specialized text like legal documents.
    nlp_pt = spacy.load("pt_core_news_lg")
except OSError:
    print("Downloading 'pt_core_news_lg' from spaCy...")
    print("This may take a few minutes...")
    spacy.cli.download("pt_core_news_lg")
    nlp_pt = spacy.load("pt_core_news_lg")

In [5]:

def preprocessing_legal_pt(text: str):
    """
    Preprocesses legal documents written in Portuguese.

    This function performs a series of cleaning and normalization steps
    tailored to the characteristics of legal texts in Portuguese.

    The pipeline is as follows:
    0.  Isolates the report by truncating after the *last* "É o relatório.".
    1.  Removes common header and footer patterns from the remaining text.
    2.  Replaces special characters, symbols, and URLs with standardized equivalents.
    3.  Standardizes common legal references (e.g., 'Art. 99' -> 'ART_99').
    4.  Tokenizes the text into individual sentences.
    5.  Processes each sentence to:
        - Lowercase words while preserving proper nouns and standardized entities.
        - Join the processed words back into a clean sentence.
    6.  Performs a final cosmetic cleanup of spacing around punctuation.

    Args:
        text: A string containing the Portuguese legal document.

    Returns:
        A list of preprocessed sentence strings.
    """

    def _replace_special_chars_pt(target: str):
        """Replaces symbols and special characters with Portuguese-centric equivalents."""
        target = re.sub(r'https?://\S+|www\.\S+', '[URL]', target)
        replace_map_pt = {
            '–': '-', '—': '-', '―': '-', '‐': '-',
            '“': '"', '”': '"', '‘': "'", '’': "'", '´': "'", '`': "'", '¨': '"',
            '…': '...', '€': ' euro ', '£': ' libra ', '$': ' dolar ',
            '¢': ' centavo ', '%': ' por cento ', '#': ' numero ', '@': ' em ',
            '•': '', '·': '', '~': '', '˚': '',
            '\t': ' ', '\r': ' ', '\ufeff': ''
        }
        for char, replacement in replace_map_pt.items():
            target = target.replace(char, replacement)
        return target

    def _standardize_legal_entities_v2(target: str) -> str:
        """
        Standardizes simple and complex legal entities using programmatic replacement.
        Handles single articles, lists ("e"), and ranges ("a").
        """
        target = target.replace('§', 'paragrafo')

        # Keyword we are looking for (singular and plural)
        keyword = "artigo"
        keywords_plural = "artigos"

        # --- 1. Handler for ranges like "Artigos 3 a 6" ---
        def expand_range(match):
            start_num = int(match.group(1))
            end_num = int(match.group(2))
            # Generate the numbers in the range
            numbers = range(start_num, end_num + 1)
            # Create the replacement string: "ARTIGO_3, ARTIGO_4, ..."
            return ', '.join([f"{keyword.upper()}_{num}" for num in numbers])

        range_pattern = re.compile(
            fr'\b{keywords_plural}\s+(\d+)\s+a\s+(\d+)\b',
            re.IGNORECASE
        )
        target = range_pattern.sub(expand_range, target)

        # --- 2. Handler for lists like "Artigos 4 e 5" or "3, 4 e 5" ---
        def expand_list(match):
            # Extract the full string of numbers and connectors, e.g., "3, 4 e 5"
            list_str = match.group(1)
            # Find all numbers in that string
            numbers = re.findall(r'\d+', list_str)
            # Create the replacement string
            return ', '.join([f"{keyword.upper()}_{num}" for num in numbers])

        list_pattern = re.compile(
            fr'\b{keywords_plural}\s+((?:\d+,\s*)*\d+\s+e\s+\d+)\b',
            re.IGNORECASE
        )
        target = list_pattern.sub(expand_list, target)

        # --- 3. Handler for simple cases like "Artigo 5" ---
        # This is your original pattern, slightly adapted.
        simple_pattern = re.compile(
            fr'\b({keyword})\.?\s*([\d./-]+)\b',
            re.IGNORECASE
        )
        target = simple_pattern.sub(
            lambda m: f"{m.group(1).upper().replace('.', '')}_{m.group(2).replace('.', '')}",
            target
        )

        return target


    def __isolate_report_section(target: str) -> str:
        """
        Isolates the report section by truncating the text after the *last*
        occurrence of the phrase "É o relatório.".

        This method is designed to capture the entire summary of facts while
        reliably excluding the subsequent vote and reasoning sections.

        Args:
            text: The full text of the legal document.

        Returns:
            The isolated report section of the text. If the marker is not found,
            it returns the original text.
        """
        # This regex finds "É o relatório" as a whole phrase, allowing for an
        # optional period at the end.
        report_end_marker = r'\bÉ o relatório\.?\b'

        # re.finditer finds all non-overlapping matches and returns an iterator.
        # We convert it to a list to easily access the last one.
        matches = list(re.finditer(report_end_marker, target, re.IGNORECASE | re.DOTALL))

        if matches:
            # If one or more matches are found, get the last one from the list.
            last_match = matches[-1]

            # Truncate the text at the end position of this last match.
            return target[:last_match.end()]
        else:
            # If the marker is never found, we return the original text.
            # This prevents accidental deletion of the entire document content.
            # You could add a warning here if you want to track such cases.
            # print("Warning: Report end marker not found. Returning original text.")
            return text

    def _join_broken_lines(target: str) -> str:
        """Joins lines that were likely broken during PDF extraction."""
        lines = target.split('\n')
        reconstructed_lines = []
        buffer = ""
        for line in lines:
            stripped_line = line.strip()
            if not stripped_line:
                continue

            # If the buffer is not empty and the current line looks like a continuation, append it.
            # Heuristic: The previous line (in the buffer) does not end with sentence-terminating punctuation.
            if buffer and not buffer.endswith(('.', '!', '?', ';', ':')):
                buffer += " " + stripped_line
            else:
                # If the buffer has content, it's a complete line/sentence.
                if buffer:
                    reconstructed_lines.append(buffer)
                buffer = stripped_line

        # Add the last buffered line
        if buffer:
            reconstructed_lines.append(buffer)

        return "\n".join(reconstructed_lines)  # Return text with sentences on new lines

    def __remove_authentication_block(target: str) -> str:
        """
        Finds and removes the multi-line document authentication block from STF texts.

        This pattern often looks like:
        http://www.stf.jus.br/portal/autenticacao/autenticarDocumento.asp sob o código
        48E5-EFAF-DDED-A9F6 e senha A040-0268-651A-9F60

        The function is designed to handle this pattern even when it's broken
        across multiple lines.

        Args:
            text: The text containing the pattern.

        Returns:
            The text with the authentication block removed.
        """
        # Regex to find the entire block, allowing for any whitespace (\s+) in between.
        auth_pattern = re.compile(
            r'https?://www\.stf\.jus\.br/portal/autenticacao/autenticarDocumento\.asp'
            r'\s+sob\s+o\s+código\s+[\w-]+\s+e\s+senha\s+[\w-]+',
            flags=re.IGNORECASE
        )

        return auth_pattern.sub('', target)

    import re

    def __remove_noise_patterns(target: str) -> str:
        """
        Remove common headers, footers, and other formatting artifacts from legal texts.

        This function uses a curated list of high-confidence regular expressions to
        clean the text line by line, minimizing the risk of removing substantive content.

        Args:
            text: The text to be cleaned.

        Returns:
            The cleaned text.
        """

        # List of high-confidence regex patterns for noise lines.
        # Each pattern is compiled for efficiency and targets a specific type of noise.
        noise_patterns = [
            # Matches document identifiers like 'HC 115002 / SP' or 'REsp 1.276.871-SP'
            # This is more specific and safer than the original.
            re.compile(r'^\s*(HC|REsp|AgRg)\s+[\d.-]+\s*/\s*\w{2}\s*$', re.IGNORECASE),

            # Matches explicit page numbers like 'Página 1 de 10' or 'fls. 2'.
            # We REMOVED the risky `|[\d\s]+` part.
            re.compile(r'^\s*(página|pag|fls)\.?\s*\d+(\s*de\s*\d+)?\s*$', re.IGNORECASE),

            # Matches section titles ONLY if they appear in ALL CAPS, which is a common
            # formatting for titles and reduces the risk of removing content words.
            re.compile(r'^\s*(RELATÓRIO|VOTO|EMENTA|ACÓRDÃO|DECISÃO)\s*$', re.IGNORECASE),

            # Matches final signature/footer lines with a date and court division.
            re.compile(r'^\s*\d{2}/\d{2}/\d{4}\s+(PRIMEIRA|SEGUNDA)\s+(TURMA|C[ÂA]MARA)\s*$', re.IGNORECASE),

            # Matches lines that ONLY contain the court division (e.g., 'SEGUNDA TURMA').
            re.compile(r'^\s*(PRIMEIRA|SEGUNDA)\s+(TURMA|C[ÂA]MARA)\s*$', re.IGNORECASE),
        ]

        # Process the text line by line
        cleaned_lines = []
        for line in target.split('\n'):
            # If a line is empty or matches any of the noise patterns, it's skipped.
            if not line.strip():
                continue
            if any(pattern.fullmatch(line.strip()) for pattern in noise_patterns):
                continue

            cleaned_lines.append(line)

        # Join the cleaned lines back into a single text block.
        return "\n".join(cleaned_lines)


    # Step 0: Truncates the document at the beginning of the reasoning/vote section.
    text = __isolate_report_section(text)

    # Step 1: Remove lines that look like headers or footers from the remaining text.
    text = __remove_noise_patterns(text)


    # Step 2: Join lines broken by PDF formatting.
    text = _join_broken_lines(text)

    # Remove URLs
    text = __remove_authentication_block(text)

    # Step 3: Replace special characters, symbols,
    pre_processed = _replace_special_chars_pt(text)

    # # Step 4: Standardize legal entities like 'Art. 99' to 'ART_99'.
    # pre_processed = _standardize_legal_entities_v2(pre_processed)

    # Step 5: Tokenize the document into sentences.
    # Note: Using a robust sentence tokenizer like NLTK or spaCy is crucial.
    splitted_sentences = sent_tokenize(pre_processed, language='portuguese')

    # Step 6: Process each sentence for intelligent lowercasing.
    processed_sentences = []
    for sentence in splitted_sentences:
        # Assuming nlp_pt is a loaded spaCy model for Portuguese
        doc = nlp_pt(sentence)
        words = []
        for token in doc:
            if token.pos_ == "PROPN" or token.text.isupper():
                words.append(token.text)
            else:
                words.append(token.text.lower())
        processed_sentence = ' '.join(words)

        # Step 7: Final cosmetic cleanup of spacing around punctuation.
        recover_chars = {
            " , ": ", ", " .": ". ", " ; ": "; ", " : ": ": ", " ! ": "! ", " ? ": "? ",
            "( ": " (", " )": ") ", " [ ": " [", " ]": "] ",
            " - ": "-", "  ": " "
        }
        for _ in range(3):
            for char, replacement in recover_chars.items():
                processed_sentence = processed_sentence.replace(char, replacement)
        processed_sentence = re.sub(r'(\w)([\(\{\[])', r'\1 \2', processed_sentence)
        processed_sentence = re.sub(r'([\)\}\]])(\w)', r'\1 \2', processed_sentence)
        processed_sentences.append(processed_sentence.strip())

    return processed_sentences

In [6]:

files = glob.glob("../data/datasets/STF_HC/train/raw/Preso/*.txt")
random.shuffle(files)
sample_legal_text = open(files[0]).read()

# Process the text using the new function
cleaned_sentences = preprocessing_legal_pt(sample_legal_text)




In [7]:
# Print the result
print("--- Original Text ---")
print(sample_legal_text)


--- Original Text ---

                                O SENHOR MINISTRO LUÍS ROBERTO BARROSO (RELATOR):
                                1.            Trata-se de habeas corpus, substitutivo do recurso ordinário
                        constitucional, impetrado contra decisão majoritária da Quinta Turma do
                        Superior Tribunal de Justiça que concedeu parcialmente o HC 179.534/AC
                        (redator para o acórdão o Ministro Honildo Amaral de Mello Castro,
                        Desembargador convocado do TJ/AP), nos termos da seguinte ementa:
                                                       “PENAL - HABEAS CORPUS - TRÁFICO DE DROGAS -
                                              IMPOSSIBILIDADE DE REAPRECIAÇÃO DE PROVAS -
                                              DOSIMETRIA - ELEVAÇÃO DA PENA-BASE ACIMA DO
                                              MÍNIMO LEGAL - QUANTIDADE E QUALIDADE DA
                                              DRO

In [8]:
print("\n--- Processed Sentences ---")
for sent in cleaned_sentences:
    print(sent)



--- Processed Sentences ---
O SENHOR MINISTRO LUÍS ROBERTO BARROSO (RELATOR): 
 1.
trata-se de habeas corpus, substitutivo do recurso ordinário constitucional, impetrado contra decisão majoritária da Quinta Turma do Superior Tribunal de Justiça que concedeu parcialmente o HC 179.534 / AC (redator para o acórdão o Ministro Honildo Amaral de Mello Castro, desembargador convocado do TJ / AP), nos termos da seguinte ementa: 
 " PENAL-HABEAS CORPUS-TRÁFICO DE DROGAS-IMPOSSIBILIDADE DE REAPRECIAÇÃO DE PROVAS-DOSIMETRIA-ELEVAÇÃO DA PENA-BASE ACIMA DO MÍNIMO LEGAL-QUANTIDADE E QUALIDADE DA DROGA APREENDIDA-PROPORCIONALIDADE-CAUSA DE AUMENTO DO ART.
40, INCISOS III E V DA LEI 11.343/06-INCIDENCIA AFASTADA PELO TRIBUNAL A QUO-ORDEM PARCIALMENTE CONCEDIDA.
1-A análise das alegações acerca de suposto equívoco da decisão condenatória em face das provas carreadas aos autos, demandaria o revolvimento do conjunto fático-probatório, inviável em sede de habeas corpus, o que impede a concessão da ordem 

# Spacy

In [9]:

text = "\n".join(cleaned_sentences)
nlp_pt = spacy.load("pt_core_news_lg")
doc = nlp_pt(text)

In [10]:
for token in doc:
    print(token.text, token.lemma_, token.pos_, token.dep_)


O o DET det
SENHOR SENHOR PROPN nsubj
MINISTRO MINISTRO NOUN ROOT
LUÍS LUÍS PROPN flat:name
ROBERTO ROBERTO PROPN flat:name
BARROSO BARROSO PROPN flat:name
( ( PUNCT punct
RELATOR RELATOR PROPN parataxis
): ): PUNCT punct

  
  SPACE dep
1 1 NUM nummod
. . PUNCT punct

 
 SPACE dep
trata-se tratar se VERB ROOT
de de ADP case
habeas habea NOUN obj
corpus corpus NOUN amod
, , PUNCT punct
substitutivo substitutivo ADJ amod
do de o ADP case
recurso recurso NOUN obl
ordinário ordinário ADJ amod
constitucional constitucional ADJ amod
, , PUNCT punct
impetrado impetrar VERB acl
contra contra ADP case
decisão decisão NOUN obl
majoritária majoritário ADJ amod
da de o ADP case
Quinta Quinta PROPN nmod
Turma Turma PROPN flat:name
do de o ADP case
Superior Superior PROPN nmod
Tribunal Tribunal PROPN flat:name
de de ADP case
Justiça Justiça PROPN nmod
que que PRON nsubj
concedeu conceder VERB acl:relcl
parcialmente parcialmente ADV advmod
o o DET det
HC HC PROPN obj
179.534 179.534 NUM nummod
/ / A

In [11]:

for ent in doc.ents:
    print(ent.text, ent.label_)

SENHOR MINISTRO LUÍS ROBERTO BARROSO MISC
RELATOR MISC
Quinta Turma do Superior LOC
Tribunal de Justiça ORG
HC LOC
AC ORG
Ministro Honildo Amaral de Mello Castro PER
TJ ORG
AP ORG
PENAL-HABEAS CORPUS-TRÁFICO DE DROGAS-IMPOSSIBILIDADE DE REAPRECIAÇÃO MISC
INCISOS III PER
TRIBUNAL LOC
QUO-ORDEM PARCIALMENTE CONCEDIDA LOC
Tribunal LOC
III MISC
V MISC
Lei MISC
Federação LOC
Sudinete Souza dos Santos PER
Maria José Costa Alves PER
Rio Branco LOC
AC LOC
Belém LOC
PA LOC
O Juízo da Comarca de Rio Branco MISC
AC ORG
c / c MISC
III MISC
V MISC
Lei MISC
III MISC
V MISC
III MISC
V MISC
Tribunal de Justiça do Acre ORG
III MISC
V MISC
Lei MISC
Superior Tribunal de Justiça ORG
Aresp MISC
AC ORG
Marco Aurélio Bellizze PER
Ministério Público LOC
V MISC
Lei MISC
reclusão4 MISC
III-a MISC
V-caracterizado LOC
Federação LOC
Distrito Federal LOC
Superior Tribunal de Justiça LOC
tráfico-5 PER
tráfico-3 MISC
Maria José Costa Alves PER
Ministério Público LOC
Joaquim Barbosa PER
Procuradoria-Geral da República

In [12]:

# Matcher example
from spacy.matcher import Matcher
matcher = Matcher(nlp_pt.vocab)
pattern = [{"LOWER": "brasil"}, {"IS_PUNCT": True, "OP": "?"}, {"LOWER": "economia"}]
matcher.add("BR_ECON", [pattern])
matches = matcher(doc)
matches

[]

In [13]:


# Save doc
doc.to_disk("mydoc.spacy")
# Load later
from spacy.tokens import Doc
doc2 = Doc(nlp_pt.vocab).from_disk("mydoc.spacy")


In [14]:
import spacy
import networkx as nx
import matplotlib.pyplot as plt
# 1. Map entity spans (multi-token) to single identifiers
ent_map = {}  # token index → entity span key
for ent in doc.ents:
    token_idxs = list(range(ent.start, ent.end))
    key = ent.text
    for i in token_idxs:
        ent_map[i] = key

# 2. Build a set of node labels
nodes = set()
for token in doc:
    if token.i in ent_map:
        nodes.add(ent_map[token.i])
    else:
        nodes.add(token.text)

In [15]:
def safe_label(label):
    return label.replace(":", "_")

In [16]:
G = nx.DiGraph()
G.add_nodes_from(nodes)

for token in doc:
    head = token.head
    if token.i == head.i:
        continue  # skip root self-loop

    src = safe_label(ent_map.get(token.i, token.text))
    tgt = safe_label(ent_map.get(head.i, head.text))
    G.add_edge(src, tgt, label=token.dep_)

In [21]:
# Install required libraries (if needed):
# pip install spacy pyvis networkx torch
# python -m spacy download pt_core_news_lg

import spacy
import networkx as nx
import torch
from pyvis.network import Network

def build_structural_graph(target, dep_label_map):
    nlp = spacy.load("pt_core_news_lg")
    doc = nlp(target)

    G = nx.MultiDiGraph()
    ent_map = {}

    # Collapse named entities into single nodes
    for ent in doc.ents:
        key = ent.text
        for i in range(ent.start, ent.end):
            ent_map[i] = key
        G.add_node(key, text=key, type="entity")

    # Add word nodes
    for token in doc:
        if token.i not in ent_map:
            G.add_node(token.text, text=token.text, type="word", position_in_sentence=token.i)

    # Add dependency edges with features
    for token in doc:
        head = token.head
        if token.i == head.i:
            continue
        src = ent_map.get(token.i, token.text)
        tgt = ent_map.get(head.i, head.text)
        feature = torch.zeros(len(dep_label_map))
        dep_idx = dep_label_map[token.dep_]
        feature[dep_idx] = 1.0
        G.add_edge(src, tgt, type="dep", feature=feature)

    return G

def visualize_structural_graph(nx_graph: nx.MultiDiGraph, dep_label_map: dict, output_filename: str):
    net = Network(height="800px", width="100%", bgcolor="#222222", font_color="white", cdn_resources="remote")
    rev_dep_map = {v: k for k, v in dep_label_map.items()}

    for node, data in nx_graph.nodes(data=True):
        t = data.get("type", "unknown")
        label = data.get("text", str(node))
        title = f"ID: {node}<br>Type: {t.capitalize()}<br>Text: {label}"
        size = 30 if t == "entity" else 10 + len(str(label))
        color = {"entity": "#f77f00", "word": "#003049"}.get(t, "#e0e0e0")
        net.add_node(node, label=label, title=title, color=color, group=t, size=size)

    edge_color_map = {'dep': '#8d99ae'}
    for u, v, data in nx_graph.edges(data=True):
        et = data.get("type")
        if not et:
            continue
        color = edge_color_map.get(et, "grey")
        width = 1
        title = f"Type: {et}"
        feature = data.get("feature")
        if isinstance(feature, torch.Tensor):
            idx = feature.argmax().item()
            dep_label = rev_dep_map.get(idx, "UNK")
            title += f"<br>Label: {dep_label}"
        net.add_edge(u, v, title=title, color=color, width=width, group=et)

    net.toggle_physics(True)
    net.show_buttons(filter_=['physics', 'nodes', 'edges'])
    net.save_graph(output_filename)



In [22]:
deps = spacy.load("pt_core_news_lg").get_pipe("parser").labels
dep_label_map = {label: idx for idx, label in enumerate(deps)}

G = build_structural_graph(text, dep_label_map)
visualize_structural_graph(G, dep_label_map, "dep_graph.html")


spacy.lang.pt.Portuguese